# Task 1 SQL Analysis

This notebook validates the Task 1 SQL answers against the local DuckDB database after the ingestion and dbt pipeline have been run.

Run first from the project root if needed:

```bash
bash scripts/run_pipeline.sh --date 2026-05-07 --full
```


In [ ]:
from pathlib import Path
import duckdb

cwd = Path.cwd().resolve()
project_root = cwd
while project_root != project_root.parent and not (project_root / "icims.duckdb").exists():
    project_root = project_root.parent

db_path = project_root / "icims.duckdb"
print(f"Using DuckDB database: {db_path}")

con = duckdb.connect(str(db_path), read_only=True)


Using DuckDB database: /Users/rajat/Documents/MyProjects/iCIMS-assignment/icims-de-assignment/icims.duckdb


## 1. How many jobs are currently open?

In [2]:
open_jobs_sql = """
SELECT
    COUNT(*) AS open_jobs
FROM main_staging.stg_jobs
WHERE status = 'OPEN';
"""

open_jobs = con.execute(open_jobs_sql).df()
open_jobs


,open_jobs
0,178


## 2. Candidates who applied to more than 3 distinct jobs

In [3]:
candidates_sql = """
SELECT
    c.candidate_id,
    c.first_name,
    c.last_name,
    c.email,
    COUNT(DISTINCT a.job_id) AS jobs_applied
FROM main_staging.stg_applications a
JOIN main_staging.stg_candidates c
    ON a.candidate_id = c.candidate_id
GROUP BY
    c.candidate_id,
    c.first_name,
    c.last_name,
    c.email
HAVING COUNT(DISTINCT a.job_id) > 3
ORDER BY
    jobs_applied DESC,
    c.candidate_id;
"""

candidates = con.execute(candidates_sql).df()
print(f"Candidates returned: {len(candidates)}")
candidates.head(10)


Candidates returned: 506


,candidate_id,first_name,last_name,email,jobs_applied
0,836d2c1f-7a49-496a-9168-0e55c2b14f24,Kenneth,Norton,campbelljean@hotmail.com,9
1,916fb778-ba8f-4e6c-a1a4-cd1971ef7e0a,Megan,Rice,gabriellesimpson@yahoo.com,9
2,23a49571-33cb-49ef-95c0-4765b51bd20f,Carlos,Scott,pittmanadam@hotmail.com,8
3,436adb9d-d1c5-4a00-8bb1-3407730ad7de,Charlotte,Ford,uthompson@gmail.com,8
4,5fa9b21a-504b-40c8-b3b9-37f29fe7e068,Kristen,Moore,barnesstephanie@hotmail.com,8
5,92d00319-20bd-4422-8b0b-81de3e612d66,Lee,Murray,welchanna@yahoo.com,8
6,a2ded1de-c302-40d5-a214-5243f5d23fc6,Brian,Turner,barrettmark@hotmail.com,8
7,b671f1b2-c9a0-49ef-913c-5c4ddde630e0,Amber,Arnold,vyu@yahoo.com,8
8,0240f59a-a638-4854-bb2b-8b80c08bd674,Gregory,Armstrong,jonathonparks@hotmail.com,7
9,381958f2-2b25-4bdf-a879-1fa47a80fdd0,Erica,Contreras,tsnyder@gmail.com,7


## 2a. Count of candidates who applied to more than 3 distinct jobs

In [4]:
candidate_count_sql = """
SELECT
    COUNT(*) AS candidates_with_more_than_3_jobs
FROM (
    SELECT
        candidate_id,
        COUNT(DISTINCT job_id) AS jobs_applied
    FROM main_staging.stg_applications
    GROUP BY candidate_id
    HAVING COUNT(DISTINCT job_id) > 3
) candidates;
"""

con.execute(candidate_count_sql).df()


,candidates_with_more_than_3_jobs
0,506


## 3. Top 5 departments by number of applications

In [5]:
top_departments_sql = """
SELECT
    j.department,
    COUNT(a.application_id) AS total_applications
FROM main_staging.stg_applications a
JOIN main_staging.stg_jobs j
    ON a.job_id = j.job_id
GROUP BY j.department
ORDER BY total_applications DESC
LIMIT 5;
"""

top_departments = con.execute(top_departments_sql).df()
top_departments


,department,total_applications
0,MARKETING,923
1,PRODUCT,810
2,ENGINEERING,789
3,SALES,761
4,FINANCE,629


## Close Connection

In [6]:
con.close()